# PrediMaint · Spark Structured Streaming : moyennes par machine (fenêtres d'une minute)

Lecture continue du topic Kafka `sensor.readings`, agrégation par machine sur des fenêtres
d'une minute, affichage dans la **console** (sink `console`).

* Paquet connecteur Kafka compatible avec **Spark 3.3.0 / Scala 2.12** :
  `org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0`
* Kafka vu depuis ce conteneur : `kafka:9092`
* Le sink `console` écrit dans la sortie du **driver** Spark, c'est-à-dire les journaux du
  conteneur : `docker compose logs -f ed-pyspark-jupyter`. La dernière section affiche aussi
  le résultat **dans le notebook** grâce au sink `memory`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

spark = (SparkSession.builder
         .appName("predimaint-moyennes-1min")
         .master("local[2]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
         .config("spark.jars.ivy", "/home/jovyan/work/.ivy2")   # cache Maven persistant
         .config("spark.sql.shuffle.partitions", "6")   # = partitions du topic
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark.version

## 1. Lecture du topic et décodage du JSON (schéma explicite)

In [ ]:
schema = StructType([
    StructField("machine_id", StringType()),
    StructField("ts", StringType()),
    StructField("temperature", DoubleType()),
    StructField("vibration", DoubleType()),
    StructField("pression", DoubleType()),
    StructField("puissance_kw", DoubleType()),
    StructField("bruit_db", DoubleType()),
])

brut = (spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", "sensor.readings")
        .option("startingOffsets", "latest")
        .option("failOnDataLoss", "false")
        .load())

mesures = (brut
           .select(F.from_json(F.col("value").cast("string"), schema).alias("m"))
           .select("m.*")
           .withColumn("event_time", F.col("ts").cast("timestamp")))
mesures.printSchema()

## 2. Moyennes par machine sur des fenêtres d'une minute (watermark de 2 minutes)

In [ ]:
moyennes = (mesures
            .withWatermark("event_time", "2 minutes")
            .groupBy(F.window("event_time", "1 minute").alias("fenetre"), "machine_id")
            .agg(F.count("*").alias("nb"),
                 F.round(F.avg("temperature"), 1).alias("temp_moy"),
                 F.round(F.max("temperature"), 1).alias("temp_max"),
                 F.round(F.avg("vibration"), 2).alias("vib_moy"),
                 F.round(F.avg("pression"), 2).alias("pres_moy"),
                 F.round(F.avg("puissance_kw"), 1).alias("kw_moy"),
                 F.round(F.avg("bruit_db"), 1).alias("db_moy"))
            .select(F.date_format("fenetre.start", "HH:mm").alias("debut"),
                    F.date_format("fenetre.end", "HH:mm").alias("fin"),
                    "machine_id", "nb", "temp_moy", "temp_max", "vib_moy",
                    "pres_moy", "kw_moy", "db_moy"))

## 3. Sortie console (toutes les 10 secondes)

Résultat visible avec `docker compose logs -f ed-pyspark-jupyter`.

In [ ]:
console = (moyennes.writeStream
           .queryName("console_moyennes")
           .outputMode("update")
           .format("console")
           .option("truncate", "false")
           .option("numRows", 50)
           .trigger(processingTime="10 seconds")
           .start())

## 4. Même résultat affiché dans le notebook (sink `memory`)

In [ ]:
import time
from IPython.display import clear_output, display

memoire = (moyennes.writeStream
           .queryName("moyennes")
           .outputMode("complete")
           .format("memory")
           .trigger(processingTime="10 seconds")
           .start())

for _ in range(12):          # environ 2 minutes d'affichage
    time.sleep(10)
    clear_output(wait=True)
    display(spark.sql("SELECT * FROM moyennes ORDER BY debut DESC, temp_max DESC").limit(25).toPandas())

## 5. Arrêt propre des requêtes

In [ ]:
for q in spark.streams.active:
    print("Arrêt de", q.name)
    q.stop()